# Dataset 2 — Historical Product Demand

**Author:** Mohd Ashraf Huzairie  
**Study:** Comparative Performance Analysis of Forecasting Models for Automated Warehouse Replenishment

This notebook presents the essential, reproducible Dataset 2 analysis without the duplicated model code and multi-megabyte logs from the original experiment.

## 1. Setup and data contract

The source CSV is expected locally and remains subject to its publisher's license.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from warehouse_forecasting.data import load_demand_series
from warehouse_forecasting.experiment import run_experiment
from warehouse_forecasting.paper_results import table as paper_table

DATA_PATH = ROOT / 'data/dataset2/Historical Product Demand.csv'
DATASET = 'historical_product_demand'
RUN_TRAINING = False
print(f'Dataset available: {DATA_PATH.exists()}')

## 2. Clean and aggregate demand

Order-demand strings are converted to numeric values, invalid dates are removed, and product/warehouse demand is aggregated into one daily series. Missing dates are interpolated chronologically.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Download Dataset 2 and place it at {DATA_PATH.relative_to(ROOT)}')
series = load_demand_series(DATA_PATH, DATASET)
display(series.describe().to_frame('daily_order_demand'))
print(f'Date range: {series.index.min().date()} to {series.index.max().date()}')
print(f'Missing values after cleaning: {series.isna().sum()}')

## 3. Demand behaviour

The log-scaled view makes extreme spikes visible without hiding lower-volume periods. Such volatility helps explain the unusually high Dataset 2 errors reported in the paper.

In [ ]:
ax = series.plot(figsize=(12, 4), alpha=.5, label='Daily order demand')
series.rolling(30, min_periods=1).median().plot(ax=ax, linewidth=2, label='30-day median')
ax.set_yscale('symlog')
ax.set(title='Dataset 2 demand and extreme variation', xlabel='Date', ylabel='Order demand (symlog)')
ax.legend(); plt.tight_layout()

## 4. Time-aware evaluation

Expanding-window folds preserve chronology. This replaces shuffled K-fold validation, which can allow future demand patterns into training data. The held-out fold is used only for final fold scoring.

In [ ]:
if RUN_TRAINING:
    summary = run_experiment(series, ['mlp', 'rbf'], ROOT / 'artifacts/dataset2', lookback=30, n_splits=5)
    display(summary.sort_values('rmse'))
else:
    print('Training skipped. Set RUN_TRAINING = True to run MLP and RBF.')

## 5. Published comparison

The following table is explicitly labeled as reported evidence rather than fresh notebook output.

In [ ]:
published = paper_table().query("dataset == 'dataset_2'").pivot(index='model', columns='metric', values='value')
display(published[['mse', 'rmse', 'mae', 'r2', 'mape']].sort_values('rmse'))

## Interpretation and limitation

All models faced substantial Dataset 2 errors. The paper associates this with magnitude spikes and irregular demand. ARIMA has the lowest reported RMSE, while GAN has the lowest MAE, RBF the lowest MAPE, and LSTM/MLP the strongest R². Consequently, the result is metric-dependent and should not be reduced to a single unqualified winner.